In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from sqlalchemy import text
from src import config as C
from src.models import MODELS, metrics

engine = C.get_engine()
feat = pd.read_sql(text("SELECT * FROM features_weekly"), engine, parse_dates=["week_start"])

# Keep only rows where lag_52 exists (week 53 onward), so no model sees half-empty features
feat = feat.dropna(subset=["lag_52"]).reset_index(drop=True)
train = feat[feat["data_split"] == "train"]
test = feat[feat["data_split"] == "test"]
print("train:", train.shape, " test:", test.shape)

train: (78000, 13)  test: (26000, 13)


In [2]:
results = {}
for name, fn in MODELS.items():
    print("running", name, "...")
    results[name] = fn(train, test)

table = pd.DataFrame({n: metrics(df) for n, df in results.items()}).T.round(4)
print(table.sort_values("WMAPE"))

running naive ...
running seasonal_naive ...
running moving_avg ...
running lightgbm ...
                 WMAPE    MAPE     RMSE    Bias
lightgbm        0.0441  0.0506  23.6888  0.0035
seasonal_naive  0.0622  0.0686  33.3353 -0.0367
naive           0.0648  0.0732  35.8404 -0.0008
moving_avg      0.0749  0.0825  42.2722 -0.0016


In [3]:
# Stack all four models into one long table matching fact_forecast
all_fc = pd.concat([df.assign(model=name) for name, df in results.items()], ignore_index=True)
all_fc = all_fc[["store", "item", "week_start", "model", "actual", "forecast"]]

with engine.begin() as con:
    con.execute(text("DELETE FROM fact_forecast"))   # makes reruns safe, no duplicates

all_fc.to_sql("fact_forecast", engine, if_exists="append", index=False,
              chunksize=10000, method="multi")

table.to_csv(C.OUT_DIR / "04_model_metrics.csv")     # result table for the README

print(pd.read_sql(text("""SELECT model, COUNT(*) AS n, ROUND(SUM(actual)) AS total_actual
                          FROM fact_forecast GROUP BY model"""), engine))

            model      n  total_actual
0        lightgbm  26000    10710031.0
1      moving_avg  26000    10710031.0
2           naive  26000    10710031.0
3  seasonal_naive  26000    10710031.0
